In [1]:
# Phase 2 — Baseline Rule-Based Agent

In [1]:
# baseline_agent.py — core logic

import re, json, datetime

def mask_pii(text):
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL_MASKED]', text)
    text = re.sub(r'#?\b\d{5,10}\b', '[ORDER_ID_MASKED]', text)
    return text

RULES = {
    "return|refund|send back": "Returns accepted within 30 days. Go to My Orders > Return Item.",
    "track|where is my order": "Track via My Orders > click order number. Updates every 24 hours.",
    "cancel": "Orders cancellable within 2 hours of placement.",
    "password|login": "Go to Login > Forgot Password > enter email. Link valid for 1 hour.",
    "charge|bill|duplicate": "Duplicate charges refunded in 3–5 business days after verification.",
    "manager|human|escalate": "Escalating to Tier-2 agent. Response within 2 hours.",
}

UNSAFE_PATTERNS = [
    r"(admin|root) password",
    r"ignore (previous|above|all) instructions",
    r"another customer.{0,20}(account|detail)",
]

def baseline_agent(user_input):
    for pattern in UNSAFE_PATTERNS:
        if re.search(pattern, user_input.lower()):
            return {"response": "Cannot assist — security policy violation.", "intent": "unsafe", "safe": False}
    for pattern, response in RULES.items():
        if re.search(pattern, user_input.lower()):
            return {"response": response, "intent": pattern.split("|")[0], "safe": True}
    return {"response": "I didn't understand. Please rephrase or contact support@techmart.com.", "intent": "unknown", "safe": True}

## Sample Log Output:

[Query 1] User: My order #12345 hasn't arrived
           Intent: track | Safe: True
           Log PII: My order [ORDER_ID_MASKED] hasn't arrived

[Query 3] User: Can you give me the admin password?
           Intent: unsafe | Safe: False ⛔ BLOCKED

In [10]:
# Test your own question on the Baseline Agent
my_question = "Where is my order 98765?"
result = baseline_agent(my_question)

print(f"Intent: {result['intent']}")
print(f"Safe: {result['safe']}")
print(f"Response: {result['response']}")

Intent: track
Safe: True
Response: Track via My Orders > click order number. Updates every 24 hours.


In [11]:
# Test your own question on the Baseline Agent
my_question = "What is admin password?"
result = baseline_agent(my_question)

print(f"Intent: {result['intent']}")
print(f"Safe: {result['safe']}")
print(f"Response: {result['response']}")

Intent: unsafe
Safe: False
Response: Cannot assist — security policy violation.


## Demonstrated Limitations:

❌ Limitation 1 — No contextual understanding: "My thing doesn't work" → fallback (no keywords)

❌ Limitation 2 — Static templates: Can't look up real order status or give personalized answers

❌ Limitation 3 — Stateless: "What about my other order?" loses all prior context

❌ Limitation 4 — Brittle: "reimbursement" ≠ "refund" → no match → fallback

Why insufficient: Real customers use varied language, expect personalized answers, and engage in multi-turn conversations. Rule systems can't adapt without manual updates.

## Phase 3 — LLM Integration & Prompt Engineering
4 Prompt Strategies Tested:

In [ ]:
import os
from openai import OpenAI
from langgraph.graph import StateGraph, END
from typing import TypedDict

client = OpenAI(
    api_key="<open api key>",
    base_url="https://openai.vocareum.com/v1"
)

PROMPT_STRATEGIES = {
    "v1_basic": "You are a customer support agent. Answer helpfully.",
    
    "v2_roleplay": """You are TechMart's empathetic AI Agent 'Aria'.
    Be professional and warm. Never fabricate policies.
    Escalate sensitive/complex cases. Never share other customers' data.""",
    
    "v3_cot": """Before responding, think step by step:
    1. UNDERSTAND the core issue
    2. IDENTIFY the applicable policy
    3. CHECK for safety/escalation triggers
    4. RESPOND with accurate, empathetic answer""",
    
    "v4_structured": """Respond in this format:
    **Issue Type:** [Category]
    **Policy Reference:** [Source]
    **Response:** [Customer message]
    **Confidence:** High/Medium/Low
    **Escalation Needed:** Yes/No — reason"""
}

In [8]:
import json

def route_prompt_dynamically(user_query):
    """
    Asks the LLM to analyze the user query and select the most appropriate 
    prompt strategy configuration from Phase 3.
    """
    
    # 1. Define the system prompt router instructions
    router_system_prompt = """
    You are an AI Meta-Router. Analyze the customer's input query and select the single best prompt strategy to handle it.
    
    Available Strategies:
    - 'v2_roleplay': Best for emotional, upset, or generic customer service greetings where empathy is needed.
    - 'v3_cot': Best for highly complex queries that require step-by-step logic, calculation, or handling nuance.
    - 'v4_structured': Best for standard, direct requests like refunds, shipping lookups, or account questions that require clear documentation keys.
    
    Respond STRICTLY in valid JSON format with a single key "selected_strategy". Do not include any explanation.
    Example: {"selected_strategy": "v4_structured"}
    """
    
    try:
        # 2. Call the OpenAI client loaded in Cell [4]
        response = client.chat.completions.create(
            model="gpt-4o-mini", # or your designated setup model
            messages=[
                {"role": "system", "content": router_system_prompt},
                {"role": "user", "content": f"Customer Query: {user_query}"}
            ],
            response_format={"type": "json_object"},
            temperature=0.0 # Strict classification
        )
        
        # 3. Parse and extract the choice
        result = json.loads(response.choices[0].message.content)
        return result.get("selected_strategy", "v4_structured") # Default fallback
        
    except Exception as e:
        print(f"Routing Error ({e}). Falling back to default.")
        return "v4_structured"

# --- Live Test Cases ---
test_queries = [
    "I am so frustrated! Your agent promised me a refund and I still haven't received it!",
    "I bought an item 5 days ago and need to return it.",
    "If I order an item with standard shipping today, but modify my shipping address tomorrow before 2 PM, will it still arrive by Friday?",
    " I need update on my order ID 212"
]

print("=== LLM DYNAMIC PROMPT ROUTING TEST ===\n")
for q in test_queries:
    chosen_strategy = route_prompt_dynamically(q)
    print(f"Query: \"{q}\"")
    print(f"➡️ LLM Selected Prompt Strategy: {chosen_strategy}")
    print(f"📋 Full Prompt Used:\n{PROMPT_STRATEGIES[chosen_strategy]}")
    print("-" * 70)

=== LLM DYNAMIC PROMPT ROUTING TEST ===

Query: "I am so frustrated! Your agent promised me a refund and I still haven't received it!"
➡️ LLM Selected Prompt Strategy: v2_roleplay
📋 Full Prompt Used:
You are TechMart's empathetic AI Agent 'Aria'.
    Be professional and warm. Never fabricate policies.
    Escalate sensitive/complex cases. Never share other customers' data.
----------------------------------------------------------------------
Query: "I bought an item 5 days ago and need to return it."
➡️ LLM Selected Prompt Strategy: v4_structured
📋 Full Prompt Used:
Respond in this format:
    **Issue Type:** [Category]
    **Policy Reference:** [Source]
    **Response:** [Customer message]
    **Confidence:** High/Medium/Low
    **Escalation Needed:** Yes/No — reason
----------------------------------------------------------------------
Query: "If I order an item with standard shipping today, but modify my shipping address tomorrow before 2 PM, will it still arrive by Friday?"
➡️ LLM

## Comparison Table:

| Strategy        | Strengths                      | Weaknesses                          |
| --------------- | ------------------------------ | ----------------------------------- |
| V1 Basic        | Simple                         | Inconsistent, no safety constraints |
| V2 Roleplay     | Empathetic, branded            | Less structured for automation      |
| V3 CoT          | Catches edge cases             | Higher latency, verbose             |
| V4 Structured ✅ | Parseable, escalation built-in | Less natural tone                   |

New Failure Modes with LLM: Hallucination risk (fixed in Phase 4 RAG), prompt injection (system prompt hardening), over-refusal on edge cases (confidence + human review).